In [1]:
import json
from pathlib import Path
from typing import List
from tqdm import tqdm

import requests

In [2]:
SA_QUESTION_FILES = {
    "DoS":   Path("DoS_sa_qa/questions/dos_sa_questions.json"),
    "Fuzzy": Path("Fuzzy_sa_qa/questions/fuzzy_sa_questions.json"),
    "Gear":  Path("Gear_sa_qa/questions/gear_sa_questions.json"),
    "RPM":   Path("RPM_sa_qa/questions/rpm_sa_questions.json"),
}

SELECTED_DATASETS = ["DoS", "Fuzzy", "Gear", "RPM"]

In [3]:
OLLAMA_BASE_URL = "http://127.0.0.1:11434"
MODEL_ID = "glm4:latest"
MODEL_TAG = MODEL_ID.replace(":", "_").replace("-", "_")

print(f"Ollama URL: {OLLAMA_BASE_URL}")
print(f"Model: {MODEL_ID}")

Ollama URL: http://127.0.0.1:11434
Model: glm4:latest


In [4]:
import re

SA_OUTPUT_TYPE = {
    "attack_ratio": "percentage",
    "attack_pattern": "short_text",
    "dominant_id": "can_id",
    "id_concentration": "short_text",
    "fps": "number",
    "traffic_shift": "short_text",
    "max_gap": "number",
    "timing_cause": "short_text",
    "dominant_var": "number",
    "payload_behavior": "short_text",
    "dlc_mode": "integer",
    "dlc_behavior": "short_text",
    "missing_count": "integer",
    "critical_behavior": "short_text",
    "out_of_range_id_count": "integer",
    "plausibility_relation": "short_text",
    "duplicate_count": "integer",
    "timestamp_cause": "short_text",
    "attack_type": "short_text",
    "evidence_pattern": "short_text",
}

SA_TYPE_HINT = {
    "percentage": "Answer with a percentage only, e.g., 12.5%.",
    "can_id": "Answer with one CAN ID only, uppercase hex format, e.g., 0x1AF.",
    "integer": "Answer with one non-negative integer only.",
    "number": "Answer with one numeric value only.",
    "short_text": "Answer with a short phrase only (1-4 words), not a full sentence.",
}


def build_sa_prompt_text(context: str, question: str, output_type: str = "short_text", strict_mode: bool = False) -> str:
    """Short-answer prompt with type-only constraints (no fixed labels)."""
    system_prompt = (
        "You are a CAN-bus anomaly analysis assistant. "
        "Read a CAN time-window log and answer one short-answer question. "
        "Think using timestamp patterns, ID frequency, gaps, DLC, payload behavior, flags, and baseline cues internally, "
        "but do not reveal reasoning. "
        "Output only the final answer on a single line. "
        "Keep it short. Do not include explanation, steps, markdown, or extra text."
    )

    type_hint = SA_TYPE_HINT.get(output_type, SA_TYPE_HINT["short_text"])
    strict_tail = (
        "\nType enforcement: strictly follow the answer type hint only."
        if strict_mode else ""
    )

    return (
        f"{system_prompt}\n\n"
        "Analyze the following CAN bus time-window log and answer the question.\n\n"
        f"CAN log:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        f"Answer type hint:\n{type_hint}{strict_tail}\n\n"
        "Final answer (one line only):"
    )


def normalize_text(text: str) -> str:
    return " ".join(text.strip().split()).lower() if text else ""


def _sanitize_answer(text: str) -> str:
    ans = text.strip().splitlines()[0].strip() if text else ""
    return ans.strip("\"'`")


def _is_valid_answer_format(ans: str, output_type: str) -> bool:
    if not ans:
        return False

    if output_type == "can_id":
        return re.fullmatch(r"0x[0-9A-F]+", ans) is not None

    if output_type == "integer":
        return re.fullmatch(r"\d+", ans) is not None

    if output_type == "percentage":
        if not ans.endswith("%"):
            return False
        try:
            float(ans[:-1])
            return True
        except Exception:
            return False

    if output_type == "number":
        try:
            float(ans)
            return True
        except Exception:
            return False

    words = ans.split()
    return 1 <= len(words) <= 4


def _extract_ollama_text(response_json: dict) -> str:
    if isinstance(response_json, dict):
        if isinstance(response_json.get("response"), str):
            return response_json["response"]
        if "message" in response_json and isinstance(response_json["message"], dict):
            content = response_json["message"].get("content")
            if isinstance(content, str):
                return content
    return ""


def query_sa_llm(context: str, question: str, sa_type: str = "", max_new_tokens: int = 24) -> str:
    output_type = SA_OUTPUT_TYPE.get(sa_type, "short_text")

    def _run_once(strict_mode: bool) -> str:
        prompt_text = build_sa_prompt_text(
            context, question, output_type=output_type, strict_mode=strict_mode
        )
        payload = {
            "model": MODEL_ID,
            "prompt": prompt_text,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": max_new_tokens,
            },
        }
        resp = requests.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json=payload,
            timeout=180,
        )
        resp.raise_for_status()
        completion = _extract_ollama_text(resp.json())
        return _sanitize_answer(completion)

    ans = _run_once(strict_mode=False)
    if _is_valid_answer_format(ans, output_type):
        return ans

    retry_ans = _run_once(strict_mode=True)
    return retry_ans if retry_ans else ans


def load_questions(path: Path) -> List[dict]:
    """Load questions from .json (list) or .jsonl."""
    if not path.exists():
        return []
    if path.suffix == ".json":
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records

In [ ]:
import re

SA_OUTPUT_TYPE = {
    "attack_ratio": "percentage",
    "attack_pattern": "short_text",
    "dominant_id": "can_id",
    "id_concentration": "short_text",
    "fps": "number",
    "traffic_shift": "short_text",
    "max_gap": "number",
    "timing_cause": "short_text",
    "dominant_var": "number",
    "payload_behavior": "short_text",
    "dlc_mode": "integer",
    "dlc_behavior": "short_text",
    "missing_count": "integer",
    "critical_behavior": "short_text",
    "out_of_range_id_count": "integer",
    "plausibility_relation": "short_text",
    "duplicate_count": "integer",
    "timestamp_cause": "short_text",
    "attack_type": "short_text",
    "evidence_pattern": "short_text",
}

SA_TYPE_HINT = {
    "percentage": "Answer with a percentage only, e.g., 12.5%.",
    "can_id": "Answer with one CAN ID only, uppercase hex format, e.g., 0x1AF.",
    "integer": "Answer with one non-negative integer only.",
    "number": "Answer with one numeric value only.",
    "short_text": "Answer with a short phrase only (1-4 words), not a full sentence.",
}


def build_sa_prompt_text(context: str, question: str, output_type: str = "short_text", strict_mode: bool = False) -> str:
    """Short-answer prompt with type-only constraints (no fixed labels)."""
    system_prompt = (
        "You are a CAN-bus anomaly analysis assistant. "
        "Read a CAN time-window log and answer one short-answer question. "
        "Think using timestamp patterns, ID frequency, gaps, DLC, payload behavior, flags, and baseline cues internally, "
        "but do not reveal reasoning. "
        "Output only the final answer on a single line. "
        "Keep it short. Do not include explanation, steps, markdown, or extra text."
    )

    type_hint = SA_TYPE_HINT.get(output_type, SA_TYPE_HINT["short_text"])
    strict_tail = (
        "\nType enforcement: strictly follow the answer type hint only."
        if strict_mode else ""
    )

    return (
        f"{system_prompt}\n\n"
        "Analyze the following CAN bus time-window log and answer the question.\n\n"
        f"CAN log:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        f"Answer type hint:\n{type_hint}{strict_tail}\n\n"
        "Final answer (one line only):"
    )


def normalize_text(text: str) -> str:
    return " ".join(text.strip().split()).lower() if text else ""


def _sanitize_answer(text: str) -> str:
    ans = text.strip().splitlines()[0].strip() if text else ""
    return ans.strip("\"'`")


def _is_valid_answer_format(ans: str, output_type: str) -> bool:
    if not ans:
        return False

    if output_type == "can_id":
        return re.fullmatch(r"0x[0-9A-F]+", ans) is not None

    if output_type == "integer":
        return re.fullmatch(r"\d+", ans) is not None

    if output_type == "percentage":
        if not ans.endswith("%"):
            return False
        try:
            float(ans[:-1])
            return True
        except Exception:
            return False

    if output_type == "number":
        try:
            float(ans)
            return True
        except Exception:
            return False

    words = ans.split()
    return 1 <= len(words) <= 4


def _extract_ollama_text(response_json: dict) -> str:
    if isinstance(response_json, dict):
        if isinstance(response_json.get("response"), str):
            return response_json["response"]
        if "message" in response_json and isinstance(response_json["message"], dict):
            content = response_json["message"].get("content")
            if isinstance(content, str):
                return content
    return ""


def query_sa_llm(context: str, question: str, sa_type: str = "", max_new_tokens: int = 24) -> str:
    output_type = SA_OUTPUT_TYPE.get(sa_type, "short_text")

    def _run_once(strict_mode: bool) -> str:
        prompt_text = build_sa_prompt_text(
            context, question, output_type=output_type, strict_mode=strict_mode
        )
        payload = {
            "model": MODEL_ID,
            "prompt": prompt_text,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": max_new_tokens,
            },
        }
        resp = requests.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json=payload,
            timeout=180,
        )
        resp.raise_for_status()
        completion = _extract_ollama_text(resp.json())
        return _sanitize_answer(completion)

    ans = _run_once(strict_mode=False)
    if _is_valid_answer_format(ans, output_type):
        return ans

    retry_ans = _run_once(strict_mode=True)
    return retry_ans if retry_ans else ans


def load_questions(path: Path) -> List[dict]:
    """Load questions from .json (list) or .jsonl."""
    if not path.exists():
        return []
    if path.suffix == ".json":
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records

In [7]:
for ds_name in SELECTED_DATASETS:
    q_path = SA_QUESTION_FILES[ds_name]
    if not q_path.exists():
        print(f"[WARN] Short-answer questions for {ds_name} not found at {q_path}, skip.")
        continue

    questions = load_questions(q_path)
    print(f"[INFO] {ds_name}: loaded {len(questions)} short-answer questions.")

    out_dir = q_path.parent.parent / "llm_answers"
    out_dir.mkdir(parents=True, exist_ok=True)
    ans_path = out_dir / f"{ds_name.lower()}_sa_answers_{MODEL_TAG}.jsonl"

    ans_path.write_text("", encoding="utf-8")

    with ans_path.open("a", encoding="utf-8") as f:
        for rec in tqdm(questions, desc=f"{ds_name} SA answering"):
            qa_id = rec.get("qa_id")
            metadata = rec.get("metadata", {})
            dataset = metadata.get("dataset", ds_name)
            context = rec["context"]
            question = rec["question"]
            gt = rec.get("ground_truth", rec.get("answer", ""))

            pred = query_sa_llm(context, question, sa_type=rec.get("sa_type", ""))

            answer_rec = {
                "qa_id": qa_id,
                "dataset": dataset,
                "sa_type": rec.get("sa_type"),
                "model": MODEL_ID,
                "llm_answer": pred,
                "ground_truth": gt,
                "is_exact_match": normalize_text(pred) == normalize_text(gt) if gt else False,
            }
            f.write(json.dumps(answer_rec, ensure_ascii=False) + "\n")

    print(f"[INFO] {ds_name}: SA answers saved to {ans_path}")

[INFO] DoS: loaded 916 short-answer questions.


DoS SA answering: 100%|██████████| 916/916 [47:54<00:00,  3.14s/it]


[INFO] DoS: SA answers saved to DoS_sa_qa\llm_answers\dos_sa_answers_glm4_latest.jsonl
[INFO] Fuzzy: loaded 959 short-answer questions.


Fuzzy SA answering: 100%|██████████| 959/959 [57:32<00:00,  3.60s/it]  


[INFO] Fuzzy: SA answers saved to Fuzzy_sa_qa\llm_answers\fuzzy_sa_answers_glm4_latest.jsonl
[INFO] Gear: loaded 1110 short-answer questions.


Gear SA answering: 100%|██████████| 1110/1110 [1:02:31<00:00,  3.38s/it]


[INFO] Gear: SA answers saved to Gear_sa_qa\llm_answers\gear_sa_answers_glm4_latest.jsonl
[INFO] RPM: loaded 1155 short-answer questions.


RPM SA answering: 100%|██████████| 1155/1155 [2:12:41<00:00,  6.89s/it]     

[INFO] RPM: SA answers saved to RPM_sa_qa\llm_answers\rpm_sa_answers_glm4_latest.jsonl


In [ ]:
# Function to generate explanation for model's own answer

def build_explanation_prompt(context: str, question: str, model_answer: str) -> str:
    """Generate prompt for explaining why the model's answer is correct."""
    system_prompt = (
        "You are a CAN-bus anomaly analysis expert. "
        "You previously answered a question based on a CAN time-window log. "
        "Now provide a brief explanation (max 10 words) of WHY your answer is correct "
        "by citing specific evidence from the CAN log. "
        "Focus on patterns: timestamp gaps, ID frequencies, payload values, DLC changes, or flags."
    )

    return (
        f"{system_prompt}\n\n"
        f"CAN log:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        f"Your Answer:\n{model_answer}\n\n"
        "Briefly explain (max 10 words) why this answer is correct:"
    )


def generate_explanation_for_answer(context: str, question: str, model_answer: str, max_tokens: int = 30) -> str:
    """Generate explanation for why model's answer is correct."""
    prompt = build_explanation_prompt(context, question, model_answer)
    
    payload = {
        "model": MODEL_ID,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.3,
            "num_predict": max_tokens,
        },
    }
    
    try:
        resp = requests.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json=payload,
            timeout=180,
        )
        resp.raise_for_status()
        explanation = _extract_ollama_text(resp.json())
        return explanation.strip()
    except Exception as e:
        print(f"[ERROR] Failed to generate explanation: {e}")
        return ""

In [ ]:
# Add explanations to existing answer files

print("[INFO] Adding explanations to existing answer files...")
print(f"[INFO] Model: {MODEL_ID}")

for ds_name in SELECTED_DATASETS:
    # Path to existing answer file
    ans_dir = Path(f"{ds_name}_sa_qa/llm_answers")
    ans_path = ans_dir / f"{ds_name.lower()}_sa_answers_{MODEL_TAG}.jsonl"
    
    if not ans_path.exists():
        print(f"[WARN] {ds_name}: Answer file not found at {ans_path}, skipping.")
        continue
    
    # Load existing answers
    existing_answers = []
    with ans_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                existing_answers.append(json.loads(line))
    
    print(f"[INFO] {ds_name}: Loaded {len(existing_answers)} existing answers")
    
    # Load questions to get context
    q_path = SA_QUESTION_FILES[ds_name]
    questions = load_questions(q_path)
    question_map = {q["qa_id"]: q for q in questions}
    
    # Add explanations
    updated_answers = []
    for ans_rec in tqdm(existing_answers, desc=f"{ds_name} adding explanations"):
        qa_id = ans_rec.get("qa_id")
        
        # Skip if explanation already exists
        if "llm_explanation" in ans_rec and ans_rec["llm_explanation"]:
            updated_answers.append(ans_rec)
            continue
        
        # Get question context
        if qa_id not in question_map:
            print(f"[WARN] QA ID {qa_id} not found in questions, skipping")
            updated_answers.append(ans_rec)
            continue
        
        q_rec = question_map[qa_id]
        context = q_rec["context"]
        question = q_rec["question"]
        model_answer = ans_rec.get("llm_answer", "")
        
        if not model_answer:
            print(f"[WARN] No llm_answer for {qa_id}, skipping explanation")
            updated_answers.append(ans_rec)
            continue
        
        # Generate explanation
        explanation = generate_explanation_for_answer(context, question, model_answer)
        
        # Update record
        ans_rec["llm_explanation"] = explanation
        updated_answers.append(ans_rec)
    
    # Write updated answers back to file
    with ans_path.open("w", encoding="utf-8") as f:
        for ans_rec in updated_answers:
            f.write(json.dumps(ans_rec, ensure_ascii=False) + "\n")
    
    print(f"[INFO] {ds_name}: Updated {len(updated_answers)} records with explanations")

print("[INFO] Done adding explanations to all datasets!")